In [0]:
import pyspark
import pyspark.sql as sql
from pyspark.sql import functions as sf
from pyspark.sql import SparkSession
import os
from pyspark.sql.functions import current_timestamp
import logging
from pyspark.sql.types import StringType

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("Bronze to Silver")

valores_lixo = ['*','**', '***', '****', '?', '.', '', ' ', '  ', ',', ';', 'NULL', 'NÃO INFORMADO']

# Filtrar apenas tabelas reais do bronze (excluir temp views)
tabelas_no_catalogo = [t for t in spark.catalog.listTables("workspace.bronze") if t.database == 'bronze']

for tabela in tabelas_no_catalogo:    
    table_name = tabela.name
    print("Lendo a tabela: " + table_name)
    df = spark.table(f"workspace.bronze.{table_name}") 
    df = df.dropDuplicates()
    for coluna in df.columns:
        if isinstance(df.schema[coluna].dataType, StringType):
            df = df.withColumn(coluna, sf.trim(sf.col(coluna)))

            df = df.withColumn(
                coluna, 
                sf.when(sf.col(coluna)
                        .isin(valores_lixo), 
                        sf.lit(None))
                        .otherwise(sf.col(coluna))
                )    
    df.createOrReplaceTempView(f"tmp_{table_name}")
    df.printSchema()
    df.show(5)

In [0]:
# Conversão de alguns tipos - padronizar
from pyspark.sql.types import IntegerType, DateType
from pyspark.sql.types import StringType

regras_1 = {
    "cenipa_aeronave": {
        "int": ["anv_motor_qtd", "aeronave_assentos"]
    },

    "cenipa_recomendacao": {
        "data": ["recomendacao_dia_feedback"] # tem valores nulos, como posso fazer o handle?
    }
     # Ideia: cenipa_ocorrencias -> podemos juntar a ocorrencia dia e hora numa variável unica numérica de timestamp
}

def padronizar_texto(col):
    com_acento = "ÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇ"
    sem_acento = "AAAAAEEEEIIIIOOOOOUUUUC"
    col_limpa = sf.upper(sf.trim(col))
    col_limpa =  sf.translate(col_limpa, com_acento, sem_acento)
    return col_limpa

for tabela in tabelas_no_catalogo:
    table_name = tabela.name
    # Skip temp views (tables that already start with tmp_)
    if table_name.startswith("tmp_"):
        continue
    df = spark.table(f"tmp_{table_name}")
    
    # Padronização de texto
    for col in df.columns:
        if isinstance(df.schema[col].dataType, StringType): 
            df = df.withColumn(col, padronizar_texto(sf.col(col)))
    
    # Aplicar regras de conversão de tipo SE a tabela estiver em regras_1
    if table_name in regras_1:
        regras_tabela = regras_1[table_name]
        
        # Regras de Inteiro (usando try_cast para lidar com valores inválidos)
        if "int" in regras_tabela:
            for col in regras_tabela["int"]:
                if col in df.columns:
                    df = df.withColumn(col, sf.expr(f"try_cast({col} as int)"))

        # Regras de Data
        if "data" in regras_tabela:
            for col in regras_tabela["data"]:
                if col in df.columns:
                    df = df.withColumn(col, sf.try_to_timestamp(sf.col(col)))
    
    
    # Salvar uma única vez com todas as transformações aplicadas
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"workspace.silver.{table_name}")
    print(f"Tabela {table_name} salva em silver com sucesso!")